# D — Scratch ResNet18 and Continual-Learning Colab Execution

This notebook reproduces the CUDA execution workflow for Part D:

1. Prepare the archive, raw images, repository, and Python dependencies in Google Colab.
2. Run the scratch ResNet18 smoke check, baseline, training-only augmentation comparison, final held-out evaluation, and offline error analysis.
3. Run the 100-class, 10-task sequential continual-learning no-replay baseline.

All datasets, checkpoints, CSVs, and figures remain in Google Drive. Do not commit generated artifacts to Git.

## Step 1 — Verify the Colab GPU

Use a CUDA runtime before continuing. The training entry points intentionally reject CPU execution.

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), "CUDA is not connected"
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## Step 2 — Mount Drive and define stable paths

The archive and all generated outputs live in Drive. Raw images are extracted into `/content` for faster local reads during the Colab session.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/COMP9517")
ARCHIVE_PATH = DRIVE_ROOT / "tiny_inat_500.tar.gz"
DATA_DIR = Path("/content/inat_data_fresh")
PROJECT_DIR = Path("/content/9517_assignment_MVP_Group")
REPOSITORY_URL = "https://github.com/dilinganye/9517_assignment_MVP_Group.git"

print("Archive:", ARCHIVE_PATH)
print("Data directory:", DATA_DIR)
print("Project directory:", PROJECT_DIR)

In [ ]:
!ls -lh {ARCHIVE_PATH}
!ls -lh {DRIVE_ROOT}

## Step 3 — Extract the raw images

The archive is about 2.6 GB. The guard checks that `train_mini` has all 25,000 expected images, not merely that the directory exists. When an interrupted extraction leaves an incomplete directory, the archive is safely extracted over it again.

In [ ]:
EXPECTED_TRAIN_IMAGES = 25000

!mkdir -p {DATA_DIR}
!CURRENT_TRAIN_IMAGES=$(find {DATA_DIR}/train_mini -type f 2>/dev/null | wc -l); if [ "$CURRENT_TRAIN_IMAGES" -lt {EXPECTED_TRAIN_IMAGES} ]; then tar -xzf {ARCHIVE_PATH} -C {DATA_DIR}; fi
!ls {DATA_DIR}
!find {DATA_DIR}/train_mini -type f | wc -l

## Step 4 — Clone or update the project

Run this before each experiment so the Colab session uses the merged `main` branch.

In [ ]:
%cd /content
!if [ ! -d {PROJECT_DIR}/.git ]; then git clone {REPOSITORY_URL} {PROJECT_DIR}; fi
%cd {PROJECT_DIR}
!git switch main
!git pull --ff-only origin main
!git rev-parse --short HEAD

## Step 5 — Install project dependencies

Colab already provides CUDA-compatible PyTorch and torchvision. Install all other project dependencies without replacing those packages.

In [ ]:
%cd {PROJECT_DIR}
!grep -v -E '^(torch|torchvision)$' requirements.txt > /tmp/requirements-no-torch.txt
!pip install -q -r /tmp/requirements-no-torch.txt

import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

## Step 6 — Scratch pipeline smoke check

This runs one epoch only. Use a fresh output directory when repeating it; a completed directory contains checkpoints and should not be overwritten.

In [ ]:
SCRATCH_OUTPUT_ROOT = DRIVE_ROOT / "outputs/scratch_resnet18"
SMOKE_OUTPUT_DIR = SCRATCH_OUTPUT_ROOT / "smoke_1epoch"

!CUDA_VISIBLE_DEVICES=0 python scripts/train_scratch_resnet18.py \
    --image-root {DATA_DIR} \
    --epochs 1 \
    --batch-size 32 \
    --num-workers 2 \
    --output-dir {SMOKE_OUTPUT_DIR}

!ls -lh {SMOKE_OUTPUT_DIR}

## Step 7 — Scratch baseline and augmentation comparison

Run the baseline and augmentation experiments in separate directories. The baseline uses deterministic resize preprocessing; the augmentation experiment enables the existing training-only flag. Both use the same held-out validation split for checkpoint selection.

In [ ]:
BASELINE_OUTPUT_DIR = SCRATCH_OUTPUT_ROOT / "baseline_v1"
AUGMENTATION_OUTPUT_DIR = SCRATCH_OUTPUT_ROOT / "augmentation_v1"

# Run this once for a new baseline.
!CUDA_VISIBLE_DEVICES=0 python scripts/train_scratch_resnet18.py \
    --image-root {DATA_DIR} \
    --epochs 20 \
    --batch-size 64 \
    --num-workers 2 \
    --output-dir {BASELINE_OUTPUT_DIR}

In [ ]:
# Use only after an interrupted baseline run saved last_checkpoint.pt.
# --epochs means additional epochs after the restored completed epoch.
# !CUDA_VISIBLE_DEVICES=0 python scripts/train_scratch_resnet18.py \
#     --image-root {DATA_DIR} \
#     --resume \
#     --epochs 5 \
#     --batch-size 64 \
#     --num-workers 2 \
#     --output-dir {BASELINE_OUTPUT_DIR}

In [ ]:
# Run this once for the controlled training-only augmentation comparison.
!CUDA_VISIBLE_DEVICES=0 python scripts/train_scratch_resnet18.py \
    --image-root {DATA_DIR} \
    --train-augmentation \
    --epochs 20 \
    --batch-size 64 \
    --num-workers 2 \
    --output-dir {AUGMENTATION_OUTPUT_DIR}

## Step 8 — Inspect local scratch outputs

The completed historical augmentation run reached validation Top-1 0.2458 at epoch 19. Read the actual Drive CSV and curve instead of manually copying values into the report.

In [ ]:
import pandas as pd
from IPython.display import Image, display

history = pd.read_csv(AUGMENTATION_OUTPUT_DIR / "history.csv")
display(history.tail(10))
display(Image(filename=str(AUGMENTATION_OUTPUT_DIR / "training_curves.png")))

## Step 9 — Evaluate the selected scratch checkpoint once on held-out test

Use the augmentation checkpoint selected on validation. This is a final held-out evaluation, not a tuning loop.

In [ ]:
FINAL_EVALUATION_DIR = SCRATCH_OUTPUT_ROOT / "final_evaluation"

!CUDA_VISIBLE_DEVICES=0 python scripts/evaluate_scratch_resnet18.py \
    --checkpoint {AUGMENTATION_OUTPUT_DIR}/best_checkpoint.pt \
    --image-root {DATA_DIR} \
    --batch-size 64 \
    --num-workers 2 \
    --output-dir {FINAL_EVALUATION_DIR}

!cat {FINAL_EVALUATION_DIR}/metrics.json

## Step 10 — Produce scratch error-analysis artifacts

This consumes saved predictions and manifests without rerunning the model. It writes per-class metrics, hardest classes, frequent confusion pairs, and a compact confusion figure beside the final evaluation outputs.

In [ ]:
!python scripts/analyze_scratch_evaluation.py \
    --predictions {FINAL_EVALUATION_DIR}/predictions.csv \
    --output-dir {FINAL_EVALUATION_DIR}

!head -21 {FINAL_EVALUATION_DIR}/most_confused_pairs.csv
!head -16 {FINAL_EVALUATION_DIR}/hardest_classes.csv

## Step 11 — Run the continual-learning no-replay baseline

This is a separate advanced experiment: fixed 100 classes, 10 sequential tasks, and a 100-way scratch output head. Each task trains only on its own 400 training images. After each task, validation accuracy is measured for every seen task; no test samples and no replay memory are used.

In [ ]:
CONTINUAL_OUTPUT_DIR = DRIVE_ROOT / "outputs/continual_100/no_replay_v1"

!CUDA_VISIBLE_DEVICES=0 python scripts/train_continual_no_replay.py \
    --image-root {DATA_DIR} \
    --output-dir {CONTINUAL_OUTPUT_DIR}

In [ ]:
# Use only after an interruption that occurred after a completed task.
# !CUDA_VISIBLE_DEVICES=0 python scripts/train_continual_no_replay.py \
#     --image-root {DATA_DIR} \
#     --resume \
#     --output-dir {CONTINUAL_OUTPUT_DIR}

## Step 12 — Inspect continual-learning validation artifacts

`task_metrics.csv` is the primary evidence for current-task, old-task, seen-task accuracy, and average forgetting. `accuracy_matrix.json` is the source for later forgetting plots. Keep this no-replay run unchanged as the control for the replay comparisons below.

In [ ]:
import json

continual_metrics = pd.read_csv(CONTINUAL_OUTPUT_DIR / "task_metrics.csv")
display(continual_metrics)

with (CONTINUAL_OUTPUT_DIR / "accuracy_matrix.json").open() as file:
    accuracy_matrix = json.load(file)

print("Evaluation split:", accuracy_matrix["split"])
print("Accuracy matrix:")
for row in accuracy_matrix["matrix"]:
    print(row)

!ls -lh {CONTINUAL_OUTPUT_DIR}

## Step 13 — Run class-balanced replay comparisons

Keep the same task map, model, 20 epochs per task, batch size, learning rate, and validation-only protocol as `no_replay_v1`. The only changed variable is the number of stored old-class training examples. Run `M=2` and `M=5` in separate directories.

In [ ]:
REPLAY_M2_OUTPUT_DIR = DRIVE_ROOT / "outputs/continual_100/replay_m2_v1"
REPLAY_M5_OUTPUT_DIR = DRIVE_ROOT / "outputs/continual_100/replay_m5_v1"

!CUDA_VISIBLE_DEVICES=0 python scripts/train_continual_replay.py \
    --image-root {DATA_DIR} \
    --memory-per-class 2 \
    --output-dir {REPLAY_M2_OUTPUT_DIR}

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_continual_replay.py \
    --image-root {DATA_DIR} \
    --memory-per-class 5 \
    --output-dir {REPLAY_M5_OUTPUT_DIR}

In [ ]:
# Use only after an interruption that occurred after a completed replay task.
# Keep the same --memory-per-class value and output directory.
# !CUDA_VISIBLE_DEVICES=0 python scripts/train_continual_replay.py \
#     --image-root {DATA_DIR} \
#     --memory-per-class 2 \
#     --resume \
#     --output-dir {REPLAY_M2_OUTPUT_DIR}

## Step 14 — Compare validation-only CL results

Compare no-replay, `M=2`, and `M=5` using validation metrics only. `memory_summary.csv` confirms the replay capacity after every completed task. Do not select a budget using held-out test data.

In [ ]:
def load_task_metrics(name, output_dir):
    metrics = pd.read_csv(output_dir / "task_metrics.csv")
    return metrics.assign(experiment=name)

comparison = pd.concat(
    [
        load_task_metrics("no_replay", CONTINUAL_OUTPUT_DIR),
        load_task_metrics("replay_m2", REPLAY_M2_OUTPUT_DIR),
        load_task_metrics("replay_m5", REPLAY_M5_OUTPUT_DIR),
    ],
    ignore_index=True,
)

display(
    comparison[
        [
            "experiment",
            "task_id",
            "current_task_accuracy",
            "old_task_accuracy",
            "seen_task_accuracy",
            "average_forgetting",
        ]
    ]
)

for name, output_dir in [("replay_m2", REPLAY_M2_OUTPUT_DIR), ("replay_m5", REPLAY_M5_OUTPUT_DIR)]:
    print(f"\n{name} memory summary")
    display(pd.read_csv(output_dir / "memory_summary.csv"))